# Dubai Urban Park — Complete Ecological Design Workflow

**Scenario:** A 2.3-hectare public park in Al Barsha South, Dubai.

**Design brief:** Maximize shade and biodiversity, minimize irrigation water use, sequester carbon, and meet Dubai Municipality landscape standards.

**Climate:** Arid desert hot (Köppen BWh). Average maximum summer temperature: 47°C. Annual rainfall: ~100mm.

**Soil:** Sandy loam with moderate salinity (4.2 dS/m).

---

## Workflow

1. Define the site
2. Load the MENA species database
3. Run Habitat Suitability — which species fit this site?
4. Evaluate Water Budget — how much irrigation is needed?
5. Assess Biodiversity — how diverse is the palette?
6. Estimate Carbon — how much CO₂ is sequestered?
7. Check Species Interactions — do these species work together?
8. Run Design Optimizer — find the optimal palette
9. Assess Urban Heat Mitigation — how much cooling do we get?

## 1. Define the Site

In [ ]:
from natureos.site import Site, SoilProfile, ClimateZone, LandUse, SoilTexture

site = Site(
    name="Al Barsha South Park",
    climate_zone=ClimateZone.BWH,
    soil=SoilProfile(
        texture=SoilTexture.SANDY_LOAM,
        salinity_dsm=4.2,
        organic_matter_pct=0.8,
        ph=7.8
    ),
    area_hectares=2.3,
    land_use=LandUse.PUBLIC_PARK,
    annual_rainfall_mm=100.0,
    max_summer_temp_c=47.0,
    latitude=25.07,
    longitude=55.20
)

print(f"Site: {site.name}")
print(f"Climate: {site.climate_zone.value} ({'Arid' if site.is_arid else 'Non-arid'})")
print(f"Soil: {site.soil.texture.value}, Salinity: {site.soil.salinity_dsm} dS/m {'(Saline)' if site.is_saline else '(Non-saline)'}")
print(f"Area: {site.area_hectares} hectares")
print(f"Max summer temp: {site.max_summer_temp_c}°C")

## 2. Load the MENA Species Database

In [ ]:
from natureos.data.mena_species import ALL_SPECIES, native_species, low_water_species

print(f"Total species in database: {len(ALL_SPECIES)}")
print(f"Native species: {len(native_species())}")
print(f"Low-water species (very_low + low): {len(low_water_species())}")
print()

for sp in ALL_SPECIES:
    print(f"  {sp.display_name:<45} Water: {sp.water_regime.value:<10} Salt: {sp.salinity_tolerance.value:<12} Heat: {sp.thermal_tolerance.value}")

## 3. Habitat Suitability — Which Species Fit This Site?

In [ ]:
from natureos.engines.habitat import HabitatSuitability, SuitabilityClass

suitability = HabitatSuitability(site)
results = suitability.evaluate_all(ALL_SPECIES)

print("Species ranked by suitability for Al Barsha South Park:\n")
for i, r in enumerate(results, 1):
    indicator = "✅" if r.suitability_class in (SuitabilityClass.HIGHLY_SUITABLE, SuitabilityClass.SUITABLE) else "⚠️" if r.suitability_class == SuitabilityClass.MODERATELY_SUITABLE else "❌"
    print(f"{i:2}. {indicator} {r.summary()}")
    print(f"      Factors: Water={r.factor_scores['water_compatibility']:.0%}, Thermal={r.factor_scores['thermal_compatibility']:.0%}, Salinity={r.factor_scores['salinity_compatibility']:.0%}, Ecosystem={r.factor_scores['ecosystem_match']:.0%}, Soil={r.factor_scores['soil_texture_match']:.0%}")
    print()

## 4. Water Budget — How Much Irrigation?

In [ ]:
from natureos.engines.water import WaterBudget

# Top 5 suitable species
top_species = [r.species for r in results[:5]]

water = WaterBudget(site=site, irrigation_efficiency=0.85)
water_result = water.calculate(top_species)

print(water_result.summary())

## 5. Biodiversity Assessment

In [ ]:
from natureos.engines.biodiversity import BiodiversityIndex

# Assume planting quantities for a realistic design
planting_plan = {
    top_species[0]: 45,   # Dominant tree
    top_species[1]: 30,   # Secondary tree
    top_species[2]: 25,   # Accent tree
    top_species[3]: 60,   # Shrub mass
    top_species[4]: 80,   # Groundcover / shrub
}

bio = BiodiversityIndex(species_abundances=planting_plan)
bio_result = bio.calculate()

print(bio_result.summary())
print(f"\nInterpretation: {bio_result.interpretation}")

## 6. Carbon Sequestration Estimate

In [ ]:
from natureos.engines.carbon import CarbonEstimator

carbon = CarbonEstimator(
    species_counts=planting_plan,
    site_area_hectares=site.area_hectares,
    ecosystem_type="urban_park",
    time_horizon_years=25
)
carbon_result = carbon.calculate()

print(carbon_result.summary())
print(f"\nThis is equivalent to removing {carbon_result.co2_equivalent_t / 4.6:.1f} passenger vehicles from the road for one year.")

## 7. Species Interactions — Do They Work Together?

In [ ]:
from natureos.engines.interactions import SpeciesInteraction, InteractionType

interaction = SpeciesInteraction(species_list=top_species)
interaction_result = interaction.analyse()

print(interaction_result.summary())
print()

if interaction_result.conflict_pairs:
    print("⚠️  Potential conflicts:")
    for pair in interaction_result.conflict_pairs:
        print(f"  • {pair.summary()} — {pair.rationale}")
else:
    print("✅ No significant conflicts detected.")

if interaction_result.facilitation_pairs:
    print("\n Facilitation relationships:")
    for pair in interaction_result.facilitation_pairs:
        print(f"  • {pair.summary()} — {pair.rationale}")

## 8. Design Optimizer — Find the Optimal Palette

In [ ]:
from natureos.engines.optimizer import DesignOptimizer, Objective

optimizer = DesignOptimizer(
    candidate_species=ALL_SPECIES,
    site=site,
    objectives=[
        Objective.MAXIMIZE_BIODIVERSITY,
        Objective.MINIMIZE_WATER,
        Objective.MAXIMIZE_CARBON,
        Objective.MINIMIZE_COST,
    ],
    palette_size_min=3,
    palette_size_max=8,
    population_size=50,
    generations=30
)

print("Running multi-objective optimization...")
opt_result = optimizer.optimize()

print(opt_result.summary())
print()

best = opt_result.best()
if best:
    print("Best design solution:")
    print(best.summary())

## 9. Urban Heat Mitigation — Cooling Effect

In [ ]:
from natureos.engines.urban_heat import UrbanHeatMitigation

# Use the optimized palette with estimated planting quantities
if best:
    heat_counts = {sp: 50 for sp in best.species}
    heat = UrbanHeatMitigation(
        species_counts=heat_counts,
        site_area_m2=site.area_hectares * 10000
    )
    heat_result = heat.assess()
    print(heat_result.summary())
    print(f"\nCanopy cover: {heat_result.canopy_cover_pct:.1f}% of the site")
    print(f"This vegetation provides cooling equivalent to {heat_result.cooling_capacity_kw:.0f} kW — roughly {heat_result.cooling_capacity_kw / 3.5:.0f} typical residential AC units.")

---

## Summary

This notebook demonstrates the complete NatureOS workflow:

1. **Site** — Defined real Dubai site conditions
2. **Species** — Loaded 11 MENA native and adapted species
3. **Habitat Suitability** — Ranked species by ecological fit
4. **Water Budget** — Estimated irrigation demand
5. **Biodiversity** — Computed diversity metrics
6. **Carbon** — Estimated 25-year sequestration
7. **Interactions** — Checked species compatibility
8. **Optimizer** — Found Pareto-optimal species palette
9. **Heat Mitigation** — Quantified cooling effect

**Next steps:** Adjust the site parameters, try different species pools, or modify the optimization objectives to explore alternative design scenarios.

---

*NatureOS Core v0.1.0 | MENA Species Dataset v0.1.0 | Apache 2.0 License*